In [ ]:
# Imports
import sys
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

In [ ]:
# CD-NLGSSM imports
from cd_dynamax import make_key_sequence
from cd_dynamax.src.utils.experiment_utils import *
from cd_dynamax.src.utils.simulation_utils import filter_and_forecast
from cd_dynamax.src.utils.demo_utils import sample_t_emissions
sys.path.append("..")
# Demo plotting import
from demo_plot_filter_forecast import (
    data_plotter,
    plot_filter_then_forecast_state_results,
)
from demo_plot_parameter_learning import (
    learnable_params_to_df,
    plot_mll_learning_curve,
    plot_param_sequences,
    plot_param_dist
)


## Define the Lorenz 63 Model


We generate data from a Lorenz 63 system, from dynamics with the following stochastic differential equations:

\begin{align*}
\frac{d x}{d t} &= a(y-x) + \sigma w_x(t) \\
\frac{d y}{d t} &= x(b-z) - y + \sigma w_y(t) \\
\frac{d z}{d t} &= xy - cz + \sigma w_z(t),
\end{align*}

With parameters $a=10, b=28, c=8/3$, the system gives rise to chaotic behavior, and we choose $\sigma=1.0$ for diffusion.

To generate data, we numerically approximate random path solutions to this SDE using Heun's method (i.e. improved Euler), as implemented in [Diffrax](https://docs.kidger.site/diffrax/api/solvers/sde_solvers/).


We assume the observation model is
\begin{align*}
y(t) &= H x(t) + r(t) \\
r(t) &\sim N(0,R),
\end{align*}
where we choose $R=I$. 

Namely, **we impose partial observability with H=[1, 0, 0]**, with noisy observations, sampled at irregular time intervals.

In [ ]:
# Default Lorenz 63 parameter definition in config file
# where only the first component (x_1) is observed
default_lorenz63_config_model = "../configs/model/l63_x1_filter"

In [ ]:
# Specify true Lorenz '63 parameters
from cd_dynamax.src.utils.physics_based_models import LearnableLorenz63_Drift

# Define the True Lorenz 63 Model parameters dictionary
true_model_def = {
    'initial_values.dynamics_drift': {
        "params": LearnableLorenz63_Drift(
            sigma=10.0,
            rho=28.0,
            beta=8.0 / 3.0
        ),
        "props": None # Let us use the default parameter properties
    }
}


In [ ]:
# Create and initialize the CD-NLGSSM model
true_model, true_params, true_props = create_cdnlgssm_model_from_config(
    true_model_config_file=default_lorenz63_config_model,
    overrides=true_model_def,
)

### Simulate data using true CDNLGSSM model

In [ ]:
# Simulate emission times
keys = make_key_sequence(0)
t_emissions = sample_t_emissions(
    start=0.0,
    stop=50.0,
    dt=0.01,
    regular=False,
    key=next(keys)
)

# Simulate data using true parameters (observed at times t_emissions)
# Note that default SDE diffeqsolve settings solve using dfx.Heun() with dfx.ConstantStepSize() and dt0=0.01 step size.
states, emissions = true_model.sample(
    params=true_params,
    key=next(keys),
    num_timesteps=len(t_emissions),
    t_emissions=t_emissions,
    transition_type="path",
)

In [ ]:
# Plot the results in a subplot with 3 rows, with observations overlaid in first emission_dims rows
fig, axes = plt.subplots(
    true_model.state_dim,
    1,
    figsize=(10, 8),
    sharex=True
)
state_labels = ['x', 'y', 'z']
for d in range(true_model.state_dim):
    ### True data
    data_plotter(
        ax=axes[d],
        t_idx=t_emissions,
        states=states[:, d],
        state_label=state_labels[d],
        state_style={'color': 'black', 'linestyle': '-', 'linewidth': 1.5},
        emissions=emissions[:, d] if d < emissions.shape[1] else None,
        emission_label='True Observation' if d < emissions.shape[1] else None,
        emissions_style={'color': 'orange', 'linestyle': 'None', 'marker': 'x', 'markersize': 3, 'alpha': 0.5},
        plot_observations=True and d < emissions.shape[1],
    )
axes[-1].set_xlabel('Time')
plt.suptitle('Lorenz \'63 System States and Observations')
plt.tight_layout()
plt.show()

## Fitting another Lorenz 63 model to the data, using MCMC

### Learnable model definition 

In [ ]:
# New Lorenz 63 model definition in config file
# where only the first component (x_1) is observed
learnable_lorenz63_config_model = "../configs/model/l63_x1_fit_to_data"

# If desired, we can override the initial parameters here
# We will simply override the prior used to draw the initial drift parameters
learnable_model_def = {
    'prior.prior_class_file': 'configs/prior/l63_mech_drift_hi_info.py'
}

In [ ]:
# Create and initialize the CD-NLGSSM model
learnable_model, learnable_params, learnable_props = create_cdnlgssm_model_from_config(
    true_model_config_file=learnable_lorenz63_config_model,
    overrides=learnable_model_def,
)

## Fitting: MCMC-based optimization of log-likelihood, computed via Extended Kalman Filter (EKF) 

Extended Kalman Filter (EKF)

In [ ]:
# Default EKF settings are defined in config file
default_ekf_config = "../configs/filter/ekf_StateFirst_EmissionsFirst"

In [ ]:
# Figure-out the filtering/smoothing settings from config
ekf_hyperparams, ekf_info = create_cdnlgssm_filter_from_config(
    default_ekf_config,
    overrides={}
)

MCMC for log-likelihood maximization:

In this case, we will resort to NUTS, a flexible MCMC algorithm that leverages gradient information

In [ ]:
# Load MCMC-NUTS fitting configuration

# Fit configuration file
fit_config_file = "../configs/fitting/nuts_l63_x1_fit_to_data"

# Read the fitting configuration
from configparser import ConfigParser
config = ConfigParser()
config.read(fit_config_file)

# For each optimization method specified in the config
optim_configs = config.sections()

In [ ]:
# Run MCMC for parameter estimation
if 'mcmc' in optim_configs:
    # MCMC configuration
    mcmc_config = config['mcmc']
    # MCMC MAP estimation via the cd-dynamax learnable_model's fit_mcmc method
    print("\nStarting MCMC MAP estimation...")
    mcmc_results = learnable_model.fit_mcmc(
        initial_params = learnable_params,
        props=learnable_props,
        emissions=emissions,
        t_emissions=t_emissions,
        filter_hyperparams=ekf_hyperparams,
        inputs=None,
        mcmc_algorithm=mcmc_config_to_dict(mcmc_config),
        verbose=mcmc_config.getboolean('verbose', True),
        key=jr.PRNGKey(mcmc_config.getint('key', 0))
    )

Organize results for easier plotting

In [ ]:
# Reformat the results into a dictionary
mcmc_warmup_param_samples = mcmc_result[0]
mcmc_param_samples = mcmc_result[1]
mcmc_warmup_log_probs = mcmc_result[2]
mcmc_log_probs = mcmc_result[3]

In [ ]:
# Plot the marginal log likelihood curve over NUTS warmup
plot_mll_learning_curve(
    true_model = true_model,
    true_params = true_params,
    true_emissions = emissions,
    t_emissions = t_emissions,
    marginal_lls = mcmc_warmup_log_probs,
    filter_hyperparams=ekf_hyperparams,
    plot_save_path=None # Plot here instead of saving
)

In [ ]:
# Plot the marginal log likelihood curve over NUTS sampling
plot_mll_learning_curve(
    true_model = true_model,
    true_params = true_params,
    true_emissions = emissions,
    t_emissions = t_emissions,
    marginal_lls = mcmc_log_probs,
    filter_hyperparams=ekf_hyperparams,
    plot_save_path=None # Plot here instead of saving
)

In [ ]:
# Plot MCMC parameters ---translating trees to dataframes
mcmc_samples_df = learnable_params_to_df(
    mcmc_param_samples, learnable_props
)
true_params_df =learnable_params_to_df(
    true_params, learnable_props
)
init_params_df = learnable_params_to_df(
    learnable_params, learnable_props
)

In [ ]:
# Plot MCMC parameter trajectories over MCMC iterations
plot_param_sequences(
    param_history=mcmc_samples_df,
    true=true_params_df,
    init=init_params_df,
    pointwise_estimate=mcmc_samples_df.mean(axis=0).to_frame().T,
    burn_in_frac=0.0,
    pairwise_plots=True,
    plot_save_path=None # Plot here instead of saving
)

In [ ]:
# Plot parameter distributions from MCMC samples
plot_param_dist(
    samples=mcmc_samples_df,
    true=true_params_df,
    init=init_params_df,
    pointwise_estimate=mcmc_samples_df.mean(axis=0).to_frame().T,
    burn_in_frac=0.0,
    pairwise_plots=True,
    plot_save_path=None # Plot here instead of saving
)

## Filter then forecast using true and learned models

We now generate a new test trajectory from the true model, then do filtering and forecasting (using both true and learned models) to evaluate short-term forecasting performance.

In [ ]:
# Define a set of time indices for filtering and forecasting
test_t_emissions = sample_t_emissions(
    start=0.0,
    stop=100.0,
    dt=0.01,
    regular=True,
    key=next(keys)
)

In [ ]:
# Now, generate a new testing trajectory from the true model. 
test_states, test_emissions = true_model.sample(
    params=true_params,
    key=next(keys),
    num_timesteps=len(test_t_emissions),
    t_emissions=test_t_emissions,
    transition_type="path",
)

In [ ]:
# Forecasting and filtering on specific range
T0 = 45.0
T_filter_end = 50.0
T_forecast_end = 52.0

### Filtering and Forecasting with the true model 

Note that the diffusion in the true SDE model (combined with the chaotic sensitivies of the Lorenz drift) makes even high-quality forecasts diverge quickly.

In [ ]:
print("Running filtering with {filter_name} with True model from T={T0} up to T={T_filter_end} and forecasting up to T={T_forecast_end}.".format(
    filter_name = ekf_info['name'] if ekf_info is not None and 'name' in ekf_info else "the specified filter",
    T0=T0, T_filter_end=T_filter_end, T_forecast_end=T_forecast_end
    )
)

# Perform filtering and forecasting
true_test_ekf_filtered, true_test_ekf_forecasted, start_idx_filter, stop_idx_filter, start_idx_forecast, stop_idx_forecast = filter_and_forecast(
    model_params=true_params,
    filter_hyperparams=ekf_hyperparams,
    t_emissions=test_t_emissions,
    emissions=test_emissions,
    T0=T0,
    T_filter_end=T_filter_end,
    T_forecast_end=T_forecast_end,
)

In [ ]:
# Plot both filtering and forecasting results
plot_filter_then_forecast_state_results(
    data={
        'states': test_states,
        'emissions': test_emissions,
        't_emissions': test_t_emissions,
    },
    results={
        'filtered': tree_to_dict(true_test_ekf_filtered),
        'forecasted': tree_to_dict(true_test_ekf_forecasted),
        'start_idx_filter': start_idx_filter,
        'stop_idx_filter': stop_idx_filter,
        'start_idx_forecast': start_idx_forecast,
        'stop_idx_forecast': stop_idx_forecast,
    },
    plot_only_filter_forecast_window=True,
    results_file=None, # Plot here, not save
    filter_info=ekf_info,
    plot_uncertainty=True,
    plot_observations=True,
    plot_mse=False,
)

### Filtering and Forecasting with the learned model 

In [ ]:
print("Running filtering with {filter_name} with Learned model from T={T0} up to T={T_filter_end} and forecasting up to T={T_forecast_end}.".format(
    filter_name = ekf_info['name'] if ekf_info is not None and 'name' in ekf_info else "the specified filter",
    T0=T0, T_filter_end=T_filter_end, T_forecast_end=T_forecast_end
    )
)

# Perform filtering and forecasting
test_ekf_filtered, test_ekf_forecasted, start_idx_filter, stop_idx_filter, start_idx_forecast, stop_idx_forecast = filter_and_forecast(
    model_params=sgd_fitted_params,
    filter_hyperparams=ekf_hyperparams,
    t_emissions=test_t_emissions,
    emissions=test_emissions,
    T0=T0,
    T_filter_end=T_filter_end,
    T_forecast_end=T_forecast_end,
)

In [ ]:
# Plot both filtering and forecasting results
plot_filter_then_forecast_state_results(
    data={
        'states': test_states,
        'emissions': test_emissions,
        't_emissions': test_t_emissions,
    },
    results={
        'filtered': tree_to_dict(test_ekf_filtered),
        'forecasted': tree_to_dict(test_ekf_forecasted),
        'start_idx_filter': start_idx_filter,
        'stop_idx_filter': stop_idx_filter,
        'start_idx_forecast': start_idx_forecast,
        'stop_idx_forecast': stop_idx_forecast,
    },
    plot_only_filter_forecast_window=True,
    results_file=None, # Plot here, not save
    filter_info=ekf_info,
    plot_uncertainty=True,
    plot_observations=True,
    plot_mse=False,
)

## Lorenz 63's chaotic behavior: differences in true Vs learned model forecasting

Next, we will look at the long-term statistical behavior of the learned vs true models by simulating a long trajectory from each and plotting histograms and others!!

In [ ]:
# Long time emissions for long simulation: training data is too short to see true attractor
long_t_emissions = jnp.arange(
    start=0.0,
    stop=1e4,
    step=0.01
).reshape(-1, 1)

In [ ]:
# Long simulation with true parameters for comparison 
true_long_states, true_long_emissions = true_model.sample(
    params=true_params,
    key=next(keys),
    num_timesteps=len(long_t_emissions),
    t_emissions=long_t_emissions,
    transition_type="path",
)

In [ ]:
# Long simulation with learned parameters
fitted_long_states, fitted_long_emissions = learnable_model.sample(
    params=sgd_fitted_params,
    key=next(keys),
    num_timesteps=len(long_t_emissions),
    t_emissions=long_t_emissions,
    transition_type="path",
)

### Chaos dynamics plotting

In [ ]:
# Let's plot some analyses: Note that we don't expect the learned 2nd and 3rd state variables to match-up at all to the true ones.
from cd_dynamax.src.utils.plotting_chaos_utils import analyze_chaotic_dynamics
analyze_chaotic_dynamics(
    true_long_states,
    fitted_long_states,
    t=long_t_emissions.squeeze(),
    drift_true=true_params.dynamics.drift.f,
    drift_learn=sgd_fitted_params.dynamics.drift.f,
    burnin_frac=0.5,
)